# Alto Adige / Südtirol — Monthly Temperature Anomaly

This notebook visualises how mean daily maximum air temperature in South Tyrol has changed across 5-year periods relative to the 20th-century average (1980–1999).

Each cell of the heatmap answers: *"How much warmer or cooler was this month in this period compared with its long-run baseline?"* Blue = cooler, red = warmer, white = no change.

In [1]:
import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go

## 1. Load data

We connect to the local DuckDB database (read-only) and pull every daily maximum air temperature measurement together with station metadata. The `LT` sensor code stands for *Lufttemperatur* (air temperature).

In [2]:
con = duckdb.connect("../data/weather.duckdb", read_only=True)

df = con.execute("""
    SELECT m.date::DATE AS date, m.daily_max, m.scode, s.name AS station, s.altitude
    FROM measurements m
    JOIN stations s USING (scode)
    WHERE m.sensor = 'LT'
""").df()

df["date"]  = pd.to_datetime(df["date"])
df["year"]  = df["date"].dt.year
df["month"] = df["date"].dt.month

print(f"{len(df):,} rows · {df['station'].nunique()} stations · "
      f"{df['date'].min().date()} → {df['date'].max().date()}")

738,874 rows · 67 stations · 1981-01-01 → 2026-05-10


## 2. Assign 5-year periods

We group years into consecutive 5-year bins (1981–1985, 1986–1990, …). Binning reduces noise from individual anomalous years and makes the long-term trend legible.

In [3]:
BIN_SIZE = 5

year_min = df["year"].min()
year_max = df["year"].max()
bins   = list(range(year_min, year_max + BIN_SIZE, BIN_SIZE))
labels = [f"{bins[i]}–{bins[i+1]-1}" for i in range(len(bins) - 1)]

df["period"] = pd.cut(df["year"], bins=bins, right=False, labels=labels)
df = df.dropna(subset=["period"])

df["period"].value_counts().sort_index()

period
1981–1985    60049
1986–1990    64979
1991–1995    81440
1996–2000    83654
2001–2005    85737
2006–2010    88536
2011–2015    94827
2016–2020    97600
2021–2025    81685
Name: count, dtype: int64

## 3. Filter stations

To make anomalies comparable across periods, we keep only stations that have at least one observation in **every (month, period) combination**. Stations with gaps would bias the average for the periods where they are missing.

In [4]:
coverage = (
    df.groupby("scode")
    .apply(lambda g: g.groupby(["month", "period"]).ngroups)
    .reset_index(name="n_combos")
)
complete_scodes = coverage.loc[coverage["n_combos"] == coverage["n_combos"].max(), "scode"].values
df_complete = df[df["scode"].isin(complete_scodes)].copy()

n_stations = len(complete_scodes)
n_years    = df_complete["year"].nunique()
print(f"{n_stations} complete stations · {n_years} years")

31 complete stations · 45 years


## 4. Compute the 20th-century baseline

The baseline is the mean daily maximum temperature for each calendar month, averaged across all complete stations and all years from 1981 through 1999. This gives us 12 reference values — one per month — that represent "normal" pre-2000 conditions.

In [5]:
BASELINE_END_YEAR = 2000

baseline = (
    df_complete[df_complete["year"] < BASELINE_END_YEAR]
    .groupby("month")["daily_max"]
    .mean()
)
baseline.index = baseline.index.astype(int)
baseline.rename("baseline_°C")

month
1      2.542806
2      4.806573
3      8.902592
4     11.896442
5     16.690820
6     20.051315
7     23.402002
8     22.799678
9     18.530817
10    13.270944
11     6.513073
12     2.469466
Name: baseline_°C, dtype: float64

## 5. Compute anomalies

For each (period, month) cell we compute the mean daily maximum across all complete stations, then subtract the 20th-century baseline for that month. Positive values mean the period was warmer than the baseline; negative means cooler.

In [6]:
period_means = (
    df_complete.groupby(["period", "month"])["daily_max"]
    .mean()
    .reset_index()
)
period_means["anomaly"] = period_means.apply(
    lambda r: r["daily_max"] - baseline[int(r["month"])], axis=1
)

pivot = period_means.pivot(index="period", columns="month", values="anomaly")
pivot.columns = pd.to_datetime(pivot.columns.astype(str), format="%m").strftime("%b")
pivot.index   = pivot.index.astype(str)

pivot

month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
period,,,,,,,,,,,,
1981–1985,-1.114802,-1.284488,-1.040741,0.135683,-1.839347,0.275738,0.257780,-0.960968,0.519063,0.414830,0.263905,-0.111121
1986–1990,0.193039,0.045894,-0.006811,-0.408797,0.855311,-0.754345,-0.345833,-0.067442,0.664055,1.089083,0.201146,0.446392
1991–1995,0.387301,0.272150,0.534397,-0.020887,0.043158,-0.174871,0.715632,1.214731,-1.023039,-0.938686,0.294260,0.053975
1996–2000,0.484479,1.190567,0.539841,0.506086,1.303659,1.281437,-1.097499,-0.054279,0.104575,-0.714287,-0.997167,-0.167669
2001–2005,-0.471374,-0.470026,0.740532,-0.173329,1.314746,2.533295,-0.143820,0.494819,-1.043068,-0.320554,0.206230,-0.368307
2006–2010,-0.685475,-0.292389,-1.377079,1.702648,0.814102,1.287668,0.717893,-1.288736,-0.557810,-0.327852,-0.133744,-1.023279
2011–2015,-0.132920,-1.033621,0.836054,1.865332,0.595522,1.455806,0.330319,0.430276,0.215097,-0.154108,1.505658,0.993281
2016–2020,-0.273129,0.490519,0.219551,2.246353,0.103457,2.558298,0.667363,0.856597,0.409850,0.104311,-0.069654,0.372344
2021–2025,-0.007458,1.831332,0.585806,0.350953,0.194271,3.163111,1.122481,1.366861,0.560737,1.339464,0.426703,0.553223


## 6. Plot

The heatmap uses a diverging red–blue colorscale centred exactly at zero, so white always means "no change from the baseline". The color range is symmetric around the largest absolute anomaly in the dataset.

In [7]:
max_abs = np.ceil(pivot.abs().values.max() * 2) / 2   # round to nearest 0.5

subtitle = (
    f"Relative to 20th-century mean (1981–1999) · "
    f"{n_stations} stations · {n_years} years of data"
)

fig = go.Figure(
    go.Heatmap(
        x=pivot.columns,
        y=pivot.index,
        z=pivot.values,
        colorscale="RdBu_r",
        zmid=0,
        zmin=-max_abs,
        zmax=max_abs,
        colorbar=dict(title="ΔT [°C]", thickness=15),
        hovertemplate=(
            "Period: <b>%{y}</b><br>"
            "Month: <b>%{x}</b><br>"
            "ΔT: <b>%{z:.2f} °C</b><extra></extra>"
        ),
    )
)

fig.update_layout(
    title=dict(
        text=(
            "South Tyrol / Südtirol / Alto Adige: Monthly Temperature Anomaly [°C]"
            f"<br><sup>{subtitle}</sup>"
        ),
        x=0.5,
        xanchor="center",
    ),
    xaxis_title="Month",
    yaxis_title="5-Year Period",
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
    height=520,
    margin=dict(t=100, b=60, l=130, r=80),
)

fig.show()